In [0]:
%sql
USE workspace.metropulse;

In [0]:
%sql
SELECT
  COUNT(*)                                          AS routes_running,
  SUM(weekday_trips)                                AS weekday_trips,
  (SELECT COUNT(*) FROM gold_stop_summary)          AS stops_served,
  SUM(CASE WHEN frequent_midday THEN 1 ELSE 0 END)  AS frequent_routes
FROM gold_route_summary;

routes_running,weekday_trips,stops_served,frequent_routes
109,9160,8791,22


In [0]:
%sql
SELECT route_short_name, route_long_name, mode, weekday_trips, stops_served, midday_headway_min
FROM gold_route_summary
ORDER BY weekday_trips DESC
LIMIT 10;

route_short_name,route_long_name,mode,weekday_trips,stops_served,midday_headway_min
700,METRORAIL RED LINE,Rail,348,50,6.0
082,Westheimer,Bus,293,195,8.0
900,METRORAIL PURPLE LINE,Rail,214,26,12.0
800,METRORAIL GREEN LINE,Rail,205,18,12.0
056,Airline / Montrose,Bus,202,209,10.0
073,Bellfort,Bus,188,99,12.0
002,Bellaire,Bus,183,150,15.0
054,Scott,Bus,172,148,15.0
004,Beechnut,Bus,172,233,15.0
025,Richmond,Bus,168,234,15.0


In [0]:
%sql
WITH ranked AS (
  SELECT route_long_name, weekday_trips,
         ROW_NUMBER() OVER (ORDER BY weekday_trips DESC) AS rnk
  FROM gold_route_summary
)
SELECT
  SUM(CASE WHEN rnk <= 10 THEN weekday_trips END)                                      AS top10_trips,
  SUM(weekday_trips)                                                                   AS all_trips,
  ROUND(100.0 * SUM(CASE WHEN rnk <= 10 THEN weekday_trips END) / SUM(weekday_trips), 1) AS top10_pct
FROM ranked;

top10_trips,all_trips,top10_pct
2145,9160,23.4


In [0]:
%sql
SELECT stop_name, routes_served, weekday_departures
FROM gold_stop_summary
ORDER BY routes_served DESC, weekday_departures DESC
LIMIT 10;

stop_name,routes_served,weekday_departures
Texas Medical Center Transit Center,16,1454
Northwest Transit Center - Platform 2,12,654
Franklin St @ Main St,12,243
Milam St @ Dallas St,11,385
Milam St @ Leeland St,10,354
Milam St @ Clay St,10,354
Milam St @ McKinney St,10,354
Milam St @ Jefferson St,10,354
Milam St @ Texas Ave,10,354
Milam St @ Rusk St,10,354


In [0]:
%sql
SELECT start_hour, mode, SUM(trips_started) AS trips_started
FROM gold_hourly_service
GROUP BY start_hour, mode
ORDER BY start_hour, mode;

start_hour,mode,trips_started
0,Bus,76
0,Rail,13
1,Bus,5
1,Rail,2
3,Bus,9
3,Rail,22
4,Bus,161
4,Rail,36
5,Bus,465
5,Rail,40


In [0]:
%sql
SELECT route_short_name, route_long_name, mode, midday_headway_min
FROM gold_route_summary
WHERE frequent_midday
ORDER BY midday_headway_min, route_short_name;

route_short_name,route_long_name,mode,midday_headway_min
700,METRORAIL RED LINE,Rail,6.0
082,Westheimer,Bus,8.0
056,Airline / Montrose,Bus,10.0
073,Bellfort,Bus,12.0
800,METRORAIL GREEN LINE,Rail,12.0
900,METRORAIL PURPLE LINE,Rail,12.0
002,Bellaire,Bus,15.0
004,Beechnut,Bus,15.0
014,Hiram Clarke,Bus,15.0
025,Richmond,Bus,15.0


In [0]:
%sql
SELECT route_short_name, route_long_name, stops_served, avg_trip_minutes
FROM gold_route_summary
ORDER BY stops_served DESC
LIMIT 10;

route_short_name,route_long_name,stops_served,avg_trip_minutes
003,Langley - Little York,292,106.3
040,Telephone / Heights,278,108.1
030,Clinton / Ella,267,82.8
006,Jensen / Greens,252,92.0
085,Antoine / Washington,250,86.4
080,MLK / Lockwood,238,73.8
025,Richmond,234,84.2
004,Beechnut,233,94.7
044,Acres Homes,230,92.2
056,Airline / Montrose,209,68.7


In [0]:
%sql
SELECT
  COUNT(DISTINCT e.stop_id)                                             AS stops_served,
  COUNT(DISTINCT CASE WHEN r.frequent_midday THEN e.stop_id END)        AS stops_with_frequent_service,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN r.frequent_midday THEN e.stop_id END)
        / COUNT(DISTINCT e.stop_id), 1)                                 AS pct_frequent
FROM silver_stop_events e
JOIN gold_route_summary r ON e.route_id = r.route_id;

stops_served,stops_with_frequent_service,pct_frequent
8791,2925,33.3
